# Búsqueda de configuración (coordenadas)

Búsqueda por coordenadas con el modelo real (`qwen17b`, T4). Mejora una ranura a la vez,
solo acepta si la precisión sube, y repite desde varios arranques al azar.

**Uso:** GPU T4. Ejecute las celdas en orden. Si Colab se desconecta, vuelva a ejecutar todo:
el caché en Drive y las semillas fijas hacen que lo ya generado no se repita.

In [ ]:
import os  # módulo estándar para consultar y crear archivos y carpetas

MARCA = "/content/.entorno_listo"  # archivo-testigo: si existe, la instalación ya se hizo en esta sesión de Colab
REPO = "https://github.com/Seloga26/ADA2026_2"  # repositorio del equipo
RAMA = "main"  # rama a usar; poner "feat/coordenadas" mientras el PR no esté integrado

if not os.path.exists(MARCA):  # solo instala la primera vez; después se salta todo el bloque
    # Líneas con "!" = comandos de shell en Colab. {RAMA} y {REPO} insertan variables de Python en el comando.
    !pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate nltk spacy emoji langdetect immutabledict
    !python -m spacy download en_core_web_sm -q
    !git clone -q https://github.com/allenai/open-instruct /content/open-instruct
    !git clone -q -b {RAMA} {REPO} /content/ADA2026_2

    import io  # buffers en memoria, para no guardar el .zip en disco
    import urllib.request  # descarga de archivos por HTTP
    import zipfile  # lectura de .zip

    NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"  # origen de los datos de nltk
    PAQUETES = [  # (carpeta, nombre) de cada paquete que necesitan los verificadores de restricciones
        ("tokenizers", "punkt"),
        ("tokenizers", "punkt_tab"),
        ("taggers", "averaged_perceptron_tagger"),
        ("taggers", "averaged_perceptron_tagger_eng"),
    ]
    for carpeta, nombre in PAQUETES:  # descarga y descomprime cada uno
        with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:  # abre la URL; "with" la cierra al terminar
            zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")  # lee los bytes y extrae donde nltk busca

    open(MARCA, "w").close()  # crea el archivo-testigo vacío: la instalación quedó lista
    print("Entorno instalado. El entorno se reinicia: vuelva a ejecutar todas las celdas.")
    os.kill(os.getpid(), 9)  # mata el proceso a propósito: Colab lo reinicia y carga las librerías recién instaladas

# Si el entorno ya existía (instalado desde otro notebook), deja el repo en la rama correcta y al día.
!cd /content/ADA2026_2 && git fetch -q origin && git checkout -q {RAMA} && git pull -q origin {RAMA}
%cd /content/ADA2026_2
print("Entorno listo")

## Parámetros (D17)

`B = 14 400` generaciones: dos arranques de hasta 2 pasadas. El caché y el historial viven en Drive.
Cada arranque se corre por separado (`ARRANQUE = 0`, luego `1`) para que una desconexión
solo cueste ese arranque.

In [ ]:
from google.colab import drive  # utilidad propia de Colab para montar Google Drive

drive.mount("/content/drive")  # pide autorización y expone Drive como carpeta
CARPETA = "/content/drive/MyDrive/oraculo_ada"  # carpeta de trabajo en Drive: sobrevive a desconexiones
os.makedirs(CARPETA, exist_ok=True)  # la crea si falta; exist_ok evita error si ya existe

ARRANQUE = 0  # 0 en la primera corrida, 1 en la segunda: cambia el punto de partida
MAX_PASADAS = 2  # tope de pasadas por arranque (D17)
B = 14_400  # presupuesto de generaciones (D17); el "_" solo separa miles para leer mejor
SEMILLA = 1  # semilla de generación del modelo (D2)
SEMILLA_BUSQUEDA = 0  # base de la semilla que elige la config inicial al azar
RUTA_CACHE = f"{CARPETA}/oraculo_cache.json"  # caché del oráculo real: lo ya generado no se repite
RUTA_HISTORIAL = f"{CARPETA}/historial_arranque_{ARRANQUE}.csv"  # un CSV de historial por arranque
# Decisión tomada con las trazas del arranque 0 (se llena DESPUÉS de leer el análisis de fallos).
ORDEN_RANURAS = None  # None = orden fijo; o p. ej. ["formato", "rol", "estrategia", "estilo", "cierre"]
OPCIONES_EXCLUIDAS = {}  # {ranura: {opciones}} que los fallos mostraron inútiles, p. ej. {"formato": {3, 5}}
MOTIVO = None  # texto corto que queda escrito en la columna decision, p. ej. "fallan las listas con formato 3"

In [ ]:
import json  # para escribir resultados en JSON

from ayudas import cargar_datos, cargar_modelo, dividir  # funciones permitidas del repositorio base
from busqueda_prompt.algoritmos import coordenadas  # nuestro algoritmo
from busqueda_prompt.contratos import OPCIONES_POR_RANURA, RANURAS  # 8 opciones y los nombres de las 5 ranuras
from oraculo import Oraculo  # oráculo real (solo funciona en Colab: necesita torch y GPU)

modelo = cargar_modelo("qwen17b")  # descarga y carga el modelo Qwen3-1.7B en la GPU
busqueda, validacion = dividir(cargar_datos())  # parte datos_visibles.json: 100 de búsqueda, 87 de validación
# cache= guarda las generaciones en Drive; lote=16 genera 16 prompts a la vez; presupuesto= es tope de memoria de GPU, NO es B
oraculo = Oraculo(modelo, busqueda, validacion, cache=RUTA_CACHE, lote=16, presupuesto=32_000_000)
print(len(busqueda), "instancias de búsqueda;", len(validacion), "de validación")  # comprobación rápida de tamaños

## Prueba corta del caché (hacerla ANTES de lanzar las 4 horas)

Comprueba dos cosas: que el archivo del caché en Drive **crece** al evaluar, y que repetir las
mismas evaluaciones con un oráculo nuevo (que lee el caché desde Drive) es **casi instantáneo**.
Son solo 10 generaciones (unos 20 s). Si el tamaño no crece o la repetición tarda lo mismo,
el caché no sirve: avisar antes de lanzar la búsqueda.

In [ ]:
import time  # medir tiempos

MUESTRA = busqueda[:10]  # solo 10 instancias: prueba rápida
config_prueba = {"rol": 2, "estrategia": 1, "formato": 4, "estilo": 1, "cierre": 7, "temperatura": 0.0}  # la misma de la medición


def tamano_cache():  # bytes del archivo de caché en Drive (0 si aún no existe)
    return os.path.getsize(RUTA_CACHE) if os.path.exists(RUTA_CACHE) else 0


antes = tamano_cache()  # tamaño antes de generar
t0 = time.perf_counter()
oraculo.evaluar(config_prueba, MUESTRA, semilla=SEMILLA)  # genera de verdad (o usa caché si ya estaba)
t_primera = time.perf_counter() - t0
despues = tamano_cache()  # tamaño justo después, SIN cerrar la sesión

# Oráculo nuevo con el mismo archivo: simula la sesión siguiente a una desconexión.
oraculo_nuevo = Oraculo(modelo, busqueda, validacion, cache=RUTA_CACHE, lote=16, presupuesto=32_000_000)
t0 = time.perf_counter()
oraculo_nuevo.evaluar(config_prueba, MUESTRA, semilla=SEMILLA)  # debería salir del caché
t_repetida = time.perf_counter() - t0

print(f"tamaño del caché: {antes} -> {despues} bytes  (crece: {despues > antes})")
print(f"primera evaluación: {t_primera:.1f} s   repetida con oráculo nuevo: {t_repetida:.1f} s")
print("CACHÉ OK" if (despues > antes and t_repetida < 0.2 * t_primera) else
      "REVISAR: si el tamaño ya era mayor que 0 porque la config estaba en caché, use otra config_prueba")

## Búsqueda

Un solo arranque por corrida: `semilla_busqueda` y el índice de arranque fijan el punto de
partida, así que repetir la celda reproduce el mismo recorrido (lo ya generado sale del caché).
Cada fila del historial se guarda en Drive al instante.

In [ ]:
# Para reanudar tras una desconexión: borre el CSV de este arranque (los pasos se repiten
# desde el caché sin costo de GPU) y vuelva a ejecutar esta celda.
historial = []  # aquí el algoritmo deja una fila por evaluación, también visible desde el notebook
resultado = coordenadas.resolver(
    oraculo, RANURAS, (OPCIONES_POR_RANURA,) * len(RANURAS), busqueda,  # (8,)*5 = (8, 8, 8, 8, 8): opciones de cada ranura
    semilla=SEMILLA, presupuesto=B, arranques=1, max_pasadas=MAX_PASADAS,  # un arranque por corrida, tope B
    semilla_busqueda=SEMILLA_BUSQUEDA * 1000 + ARRANQUE,  # semilla distinta por arranque, pero fija y reproducible
    historial=historial, ruta_historial=RUTA_HISTORIAL,  # memoria y CSV en Drive
    orden_ranuras=ORDEN_RANURAS, opciones_excluidas=OPCIONES_EXCLUIDAS, motivo=MOTIVO,  # decisión basada en trazas
    mostrar=lambda f: print(f"{f['paso']:4d} {f['ranura']:>10} {f['opcion']!s:>2} "  # lambda: función anónima que imprime cada fila
                            f"{f['precision']:6.1%} mejor {f['mejor_precision']:6.1%} "  # :6.1% muestra la fracción como porcentaje
                            f"gen {f['consultas']:6d} {f['decision']}"),  # !s pasa a texto; :>10 alinea a la derecha
)
print("
Ganadora del arranque:", json.dumps(resultado.ganadora))  # json.dumps muestra la config como texto JSON
print(f"Precisión: {resultado.precision:.1%}   generaciones nuevas: {resultado.consultas}")  # resumen final

In [ ]:
# Resumen para anotar en la bitácora y comparar con el otro arranque.
resumen = {"arranque": ARRANQUE, "config": resultado.ganadora,  # diccionario con lo esencial del arranque
           "precision": resultado.precision, "consultas": resultado.consultas}
with open(f"{CARPETA}/resumen_arranque_{ARRANQUE}.json", "w", encoding="utf-8") as f:  # "w" crea o reescribe; "with" cierra solo
    json.dump(resumen, f, indent=1)  # escribe el diccionario en el archivo, con sangría de 1 espacio
print(resumen)  # lo muestra también en pantalla

## Análisis de fallos (D6): léelo después del arranque 0

Cuenta en cuántas instancias falló la configuración ganadora, por familia de restricción.
Con esto se decide `ORDEN_RANURAS`, `OPCIONES_EXCLUIDAS` y `MOTIVO` (celda de parámetros) y se
corre el arranque 1. Para cada fallo citado en el informe: qué opción se descartó y qué cambió.

In [ ]:
from collections import Counter  # contador de repeticiones

r = oraculo.evaluar(resultado.ganadora, busqueda, semilla=SEMILLA)  # sale del caché: ya se midió al buscar
familia_de = {x["id"]: x["familia"] for x in busqueda}  # id de instancia -> familia de su restricción
fallos_por_familia = Counter(familia_de[t["id"]] for t in r.trazas)  # cuenta fallos por familia
total_familia = Counter(x["familia"] for x in busqueda)  # cuántas instancias hay de cada familia

print(f"Fallos: {len(r.trazas)} de {len(busqueda)}")
for fam, n in fallos_por_familia.most_common():  # de la familia que más falla a la que menos
    print(f"{fam:55s} {n:3d} fallos de {total_familia[fam]:3d}")

print("\nPrimera restricción violada (campo violo), top 10:")
for v, n in Counter(t["violo"] for t in r.trazas).most_common(10):
    print(f"{n:3d}  {v}")

print("\nEjemplos de salida fallida:")
for t in r.trazas[:3]:  # solo 3 para leer
    print("violó:", t["violo"])
    print(t["salida"][:300])
    print("-" * 60)